# PRIMA PROVA RANDOM FOREST

---

Leggo il nostro dict in `.pkl`

In [1]:
import pickle as pkl

with open('data/ottani_NIR.pkl', 'rb') as f:
    dizionario = pkl.load(f)
    
# dati train
train_ones = dizionario['train']['NIR']['ones']
train_zeros = dizionario['train']['NIR']['zeros']
train_labels = dizionario['train']['labels']

# dati test
test_ones = dizionario['test']['NIR']['ones']
test_zeros = dizionario['test']['NIR']['zeros']
test_labels = dizionario['test']['labels']


In [2]:
import numpy as np

train_x = np.concatenate((train_zeros, train_ones))
test_x = np.concatenate((test_zeros,test_ones))


---

## SMOOTHING 

provo a filtrare i dati con Savitzky-Golay

In [3]:
from scipy.signal import savgol_filter

window_size = 10
poly_order = 3

for i in range(train_x.shape[0]):
    train_x[i] = savgol_filter(train_x[i], window_size, poly_order)

for j in range(test_x.shape[0]):
    test_x[j] = savgol_filter(test_x[j], window_size, poly_order)

---

In [4]:
from sklearn.decomposition import PCA
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler

from sklearn.ensemble import RandomForestClassifier as RFC

In [ ]:
rkf = RepeatedStratifiedKFold(n_splits=8, n_repeats=10, random_state=42)

# PCA
N_COMPONENTS_OPTIONS = [2,5,7, None]
# ESTIMATOR
N_ESTIMATOR_OPTIONS = [50, 100, 250]
OBJ_FUNCTION_OPTIONS = ['gini', 'entropy']
MAX_FEATURES_OPTIONS = ["sqrt", "log2"]
MAX_DEPTH_OPTIONS = [2, 3, 5]
MIN_SAMPLES_LEAF_OPTIONS = [1, 2, 5]

# 1. Definizione Pipeline #
pipe = Pipeline([
    # Step 1: Scaling
    ("scaling", StandardScaler()),       
    
    # Step 2: Riduzione dimensionalità (PCA)
    ("reduce_dim", PCA(random_state=42)),
    
    # Step 3: Classificatore BOOTSTRAP SEMPRE TRUE PERCHÉ HO POCHI SAMPLES (44)
    ("classify", RFC(random_state=42, bootstrap=True)) 
])

# 2. Definizione griglia dei parametri #
param_grid = [
{
    # Per provare con diversi numeri di componenti (PCA)
    "reduce_dim__n_components": N_COMPONENTS_OPTIONS, 
    
    # Per provare parametri del classificatore
    "classify__n_estimators": N_ESTIMATOR_OPTIONS,
    
    # Per provare diverse objective funcions
    "classify__criterion": OBJ_FUNCTION_OPTIONS,
    
    # Diversi max features
    "classify__max_features": MAX_FEATURES_OPTIONS,
    
    "classify__max_depth": MAX_DEPTH_OPTIONS, # Ferma la crescita dell'albero oltre un certo limite
    
    "classify__min_samples_leaf": MIN_SAMPLES_LEAF_OPTIONS, # Stabilizza le foglie
},
{
    # Per saltare la PCA
    "reduce_dim": ['passthrough'], 
    
    # Per provare parametri del classificatore
    "classify__n_estimators": N_ESTIMATOR_OPTIONS,
    
    # Per provare diverse objective funcions
    "classify__criterion": OBJ_FUNCTION_OPTIONS,
    
    # Diversi max features
    "classify__max_features": MAX_FEATURES_OPTIONS,
    
    "classify__max_depth": MAX_DEPTH_OPTIONS, # Ferma la crescita dell'albero oltre un certo limite
    
    "classify__min_samples_leaf": MIN_SAMPLES_LEAF_OPTIONS, # Stabilizza le foglie
}]

# 3. Configurazione GridSearch #
grid = GridSearchCV(
    pipe, 
    param_grid=param_grid, 
    cv=rkf,
    n_jobs=-1, # «Number of jobs to run in parallel. -1 means using all processors»
    scoring={
        'score': 'accuracy',
        'sensitivity': 'recall'  # recall = sensitivity
    },
    refit='score', # «For multiple metric evaluation, needs to be a str denoting the
    # scorer to use to find the best parameters for refitting the estimator at the end»
    return_train_score=False
)

# 4. Training e Validation (su Segnale B) #
grid.fit(train_x, train_labels)

# 5. Risultati #
print(f"La miglior configurazione: {grid.best_params_}")
print(f"Fornisce accuracy in validation: {grid.best_score_:.4f}")

# 6. Test su segnale A #
accuracy_finale = grid.score(test_x, test_labels)
print(f"Risultato sul set indipendente (Segnale A): {accuracy_finale:.4f}")

La miglior configurazione: {'classify__criterion': 'entropy', 'classify__max_depth': 5, 'classify__max_features': 'log2', 'classify__min_samples_leaf': 1, 'classify__n_estimators': 100, 'reduce_dim__n_components': None}
Fornisce accuracy in validation: 0.8854
Risultato sul set indipendente (Segnale A): 1.0000


In [6]:
import pandas as pd
# Conversione dei risultati in DataFrame
results_df = pd.DataFrame(grid.cv_results_)
results_df.to_pickle("results/results_rf_GridSearch.pkl")

In [7]:
import pandas as pd

results_df = pd.DataFrame(grid.cv_results_)
# Ciascuna combinazione di parametri è una riga
print(f"Numero totale di configurazioni provate: {results_df.shape[0]}")

# Selezioniamo solo le colonne interessanti per pulire la vista
columns_to_show = [
    'param_reduce_dim__n_components', 
    'param_classify__n_estimators', 
    'param_classify__criterion', 
    'param_classify__max_features', 
    'param_classify__max_depth',
    'param_classify__min_samples_leaf',
    'mean_test_score', 
    'std_test_score', 
    'mean_test_sensitivity',
    'rank_test_score'
]

# Ordiniamo per classifica (rank_test_score)
analysis = results_df[columns_to_show].sort_values('rank_test_score')

# Se si ha, è comodo aprire analysis in un viewer tipo Data Wrangler
analysis.head(20)

Numero totale di configurazioni provate: 432


,param_reduce_dim__n_components,param_classify__n_estimators,param_classify__criterion,param_classify__max_features,param_classify__max_depth,param_classify__min_samples_leaf,mean_test_score,std_test_score,mean_test_sensitivity,rank_test_score
407,None,250,entropy,log2,5,1,0.885417,0.128002,0.937500,1
403,None,100,entropy,log2,5,1,0.885417,0.130687,0.941667,1
146,7,50,gini,sqrt,5,1,0.883333,0.127475,0.945833,3
182,7,50,gini,log2,5,1,0.883333,0.127475,0.945833,3
155,None,250,gini,sqrt,5,1,0.881250,0.132271,0.929167,5
115,None,100,gini,log2,3,1,0.881250,0.142385,0.950000,5
83,None,250,gini,sqrt,3,1,0.879167,0.136867,0.929167,7
79,None,100,gini,sqrt,3,1,0.879167,0.139381,0.920833,8
150,7,100,gini,sqrt,5,1,0.877083,0.122881,0.945833,9
191,None,250,gini,log2,5,1,0.877083,0.128408,0.920833,9
